In [ ]:
import requests
import boto3
import json
import os
import gzip
import io

In [ ]:
os.environ.setdefault('AWS_DEFAULT_PROFILE', 'Govind')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

In [ ]:
kinesis=boto3.client('kinesis')

In [ ]:
Stream_name='gharchive-stream'

In [ ]:
def get_url(date, hour): # date format is to be in yyyy-mm-dd-h
    file_name = f"{date}-{hour}.json.gz"
    url = f"https://data.gharchive.org/{file_name}"
    return url    

In [ ]:
def stream_from_gharchive(date, hour):
    url = get_url(date, hour)
    response = requests.get(url, stream=True) #stream parameter controls how the response content is downloaded

    compressed = io.BytesIO(response.raw.read())  #decompress the gzip file as it is streamed-received in chunks
    # iter the content using response.iter_content()
    #response.content loads everythong at once into memory and hence we are using response.raw.read()

    with gzip.open(compressed, 'rt', encoding='utf-8') as f:
        for line in f:
            event=json.loads(line)
            yield event    #yeild one event at a time
            

In [ ]:
def send_to_kinesis(date, hour):
    url = get_url(date, hour)
    print(f"Streaming from {url}")

    for event in stream_from_gharchive(date, hour):
        kinesis.put_record(
                StreamName=Stream_name,
                Data=json.dumps(event),
                PartitionKey=event.get('type', 'default')
            )
        time.sleep(0.1)

In [ ]:
url = get_url('2026-01-01', '23')
url

In [ ]:
response=stream_from_gharchive(url)
response

In [ ]:
response.content  # this will try to donwload the whole gz file (43 MB) and the notebook will freeze.

In [ ]:
send_to_kinesis('2026-01-01', '23')